# NequIP Tutorial: Aluminum Bulk, Al(111) Surface, and Al Adatom

In this tutorial we:

Generate a small training dataset for Aluminum using the EMT potential, consisting of:

* bulk fcc Al configurations from MD,

* a relaxed Al(111) surface slab,

* an Al adatom on the Al(111) surface.

Then, train a small NequIP model on these structures.

After training, we use the NequIP model as an ASE calculator to test its behavior by:

* running short MD on bulk Al to check dynamical stability,

* relaxing an Al(111) surface to verify interpolation near training data,

* relaxing an Al(111) slab with an Al adatom to examine how the model handles slightly perturbed surface environments.

## 2.0 Reset (Optional)

In [ ]:
# Use this only when starting a FULL new workflow run.

!rm -rf processed_dataset_*
!rm -rf al_nequip_run
!rm -rf data
!rm -f config.yaml

print("Ready for a fresh run.")


## 2.1 Install dependencies *italicised text*

In this step we install the essential Python packages required for generating atomic structures, running molecular dynamics, and training a NequIP machine-learning interatomic potential.
* NumPy for numerical operations,
* ASE (Atomic Simulation Environment) to build structures and run MD/relaxations,
* PyTorch as for machine-learning computations.
* NequIP is the main package that trains and deploys equivariant neural network potentials.

Together, these libraries provide everything needed to create a dataset, train a model, and test its predictions on aluminum systems.



> **Warning**: We install specific versions of these packages because NequIP is sensitive to version mismatches.

In [ ]:
!pip install numpy==1.26.4 --no-deps -q

In [ ]:
pip install "ase==3.22.1"

In [ ]:
!pip install "nequip==0.6.2" -q


In [ ]:
!pip install "torch==2.5.1"

In [ ]:
import nequip
print(nequip.__version__)

## 2.2 Imports and basic setup

Brief explanation of the imported packages

* `NumPy (numpy)` – Used for numerical operations, array handling, and random number generation.

* `ASE (Atomic Simulation Environment)` bulk, fcc111, add_adsorbate – Build bulk crystals, surface slabs, and adsorbate systems.

* `EMT` – A fast empirical calculator to generate example energies and forces.

* `write, read` – Save and load atomic configurations in different formats.

* `Langevin` – Run molecular dynamics simulations.

* `BFGS` – Perform structure relaxation (geometry optimization).

* `units` – Provides physical units (eV, Å, fs, etc.).

* `os` – Manage folders and file paths for saving data.

* `yaml` – Read and write configuration files required by NequIP.

In [ ]:
import numpy as np
print(np.__version__)

from ase.build import bulk, fcc111, add_adsorbate
from ase.calculators.emt import EMT
from ase.io import write, read
from ase.md.langevin import Langevin
from ase.optimize import BFGS
from ase import units

import os, yaml


This step checks whether a GPU (CUDA) is available and selects it for computation. Otherwise, it falls back to the CPU.

Also, make sure to enable GPU in Google Colab

* Go to the menu:
Runtime → Change runtime type

* Under Hardware accelerator, select GPU

* Click Save


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## 2.3 Generate Aluminum bulk, Al(111) slab, and Al adatom data (EMT labels)

In this step we generate a small training set for Aluminum using the **Effective Medium Theory (EMT)** potential as a reference calculator.  
Although EMT is not a high-accuracy model, it is fast and ideal for building demonstration datasets for NequIP. We include three types of configurations:

### **Bulk fcc Aluminum (MD sampling)**  
A 2×2×2 fcc supercell (32 atoms) is simulated using **Langevin molecular dynamics**:

- **Timestep:** 1 fs — stable choice for metallic systems  
- **Temperature:** 600 K — increases vibrational diversity  
- **Friction:** 0.02 — light damping to maintain temperature  
- **Sampling:** every 10 steps (≈10 fs) — reduces correlation between frames  

For each sampled structure we store:
- `total_energy` in `atoms.info`  

> Use `total_energy` tag and not `energy`!!!

- `forces` in `atoms.arrays`  

This yields a small but diverse set of bulk configurations.

### **Relaxed Al(111) Surface Slab**  
A 3×3×4 Al(111) slab with 8 Å vacuum is constructed and relaxed using the BFGS optimizer with the EMT calculator. This provides surface geometries where atoms are undercoordinated compared to the bulk.


### **Al Adatom on Al(111)**  
To introduce adsorption-like environments, a single Al atom is placed on an ontop site of the slab and the entire system is relaxed.  


### **Purpose of This Dataset**

- demonstrate how to build reference data using **ASE + EMT**,  
- include a range of atomic environments (bulk, surface, adatom),  
- prepare a lightweight dataset for training a NequIP model,  
- enable the model to interpolate simple Aluminum structures.

The next step is to train NequIP on these structures and then use the trained model as an ASE calculator.


In [ ]:
import numpy as np
from ase.build import bulk, fcc111, add_adsorbate
from ase.calculators.emt import EMT
from ase.io import write, read
from ase.md.langevin import Langevin
from ase.optimize import BFGS
from ase import units
import os

np.random.seed(0)
frames = []

# ----------------- SETTINGS -----------------
N_STEPS = 10_000       # total MD steps
SAMPLE_EVERY = 10      # collect 1 frame every this many steps
TEMPERATURE = 600      # K
# --------------------------------------------

# ---------- 3.1 Bulk fcc Al MD with EMT ----------
bulk_al = bulk("Al", "fcc", a=4.05).repeat((2, 2, 2))
bulk_al.calc = EMT()

dyn_bulk = Langevin(
    bulk_al,
    timestep=1.0 * units.fs,
    temperature_K=TEMPERATURE,
    friction=0.02
)

def collect_frame(atoms):
    a = atoms.copy()
    # store total_energy & forces explicitly
    E = atoms.get_potential_energy()
    F = atoms.get_forces()
    a.info["total_energy"] = float(E)
    a.arrays["forces"]     = F
    frames.append(a)

for step in range(N_STEPS):
    dyn_bulk.run(1)
    if step % SAMPLE_EVERY == 0:
        collect_frame(bulk_al)

print(f"Collected {len(frames)} bulk Al frames.")

# ---------- 3.2 Relaxed Al(111) surface slab ----------
slab = fcc111("Al", size=(3, 3, 4), a=4.05, vacuum=8.0)
slab.calc = EMT()

opt_slab = BFGS(slab, logfile=None)
opt_slab.run(fmax=0.05)

slab.info["total_energy"] = slab.get_potential_energy()
slab.arrays["forces"]     = slab.get_forces()
frames.append(slab.copy())

print("Added relaxed Al(111) slab frame.")

# ---------- 3.3 Al adatom on Al(111) surface ----------
slab_adatom = slab.copy()
add_adsorbate(slab_adatom, "Al", height=2.0, position="ontop")
slab_adatom.center(axis=2)
slab_adatom.calc = EMT()

opt_ad = BFGS(slab_adatom, logfile=None)
opt_ad.run(fmax=0.05)

slab_adatom.info["total_energy"] = slab_adatom.get_potential_energy()
slab_adatom.arrays["forces"]     = slab_adatom.get_forces()
frames.append(slab_adatom.copy())

print("Added relaxed Al adatom on Al(111) slab frame.")
print(f"Total frames (bulk + slab + adatom): {len(frames)}")

# ---------- CLEAN UP BEFORE WRITING ----------
for at in frames:
    at.info.pop("adsorbate_info", None)
    at.calc = None  # detach calculators

# ---------- SHUFFLE & SPLIT ----------
np.random.seed(0)
np.random.shuffle(frames)

n = len(frames)
split = int(0.8 * n)
train_frames = frames[:split]
val_frames   = frames[split:]

print(f"Wrote {len(train_frames)} train frames")
print(f"Wrote {len(val_frames)} validation frames")

# ---------- WRITE TO FILE ----------
os.makedirs("data", exist_ok=True)
write("data/al_train.extxyz", train_frames, format="extxyz")
write("data/al_val.extxyz",   val_frames,   format="extxyz")
print("Wrote data/al_train.extxyz and data/al_val.extxyz")

# ---------- QUICK CHECK ----------
frames2 = read("data/al_train.extxyz", ":")

print("Read back:", len(frames2), "train frames")
print("info keys of first frame:", frames2[0].info.keys())
print("array keys of first frame:", frames2[0].arrays.keys())

print("Total_energy?", "total_energy" in frames2[0].info)
print("Forces?", "forces" in frames2[0].arrays)


The dataset creation process completed successfully, producing a mixed set of Aluminum configurations for NequIP training.

### **Frame Collection**
- **1000 bulk MD frames** were generated using Langevin dynamics at 600 K.  
  This provides vibrational diversity and sampling of multiple atomic environments.
- **1 relaxed Al(111) slab frame** was added.
- **1 relaxed Al(111) slab with an Al adatom** was added.

Total frames collected: **1002**

> Although the slab and adatom categories contain only a single relaxed frame each, they are included to demonstrate the full workflow. The dataset can easily be expanded in future work to improve the model’s accuracy on surface and adsorption environments.


### **Train/Validation Split**
The dataset was shuffled and split into:
- **801 training frames**
- **201 validation frames**

Both subsets were written to:
- `data/al_train.extxyz`
- `data/al_val.extxyz`



### **Verification of Stored Data**
When reading back the training set, the first frame contains:

#### **Info keys**
`['total_energy']`  
→ Per-structure metadata such as total potential energy (required for NequIP).

#### **Array keys**
`['numbers', 'positions', 'momenta', 'forces']`  
→ Per-atom quantities needed for machine-learning force fields:
- `numbers`: atomic numbers  
- `positions`: atomic coordinates  
- `momenta`: MD momenta  
- `forces`: reference forces from EMT



In the next steps, we perform a quick check on the generated datasets to ensure data consistency. We load all training and validation frames and scan through them to verify that each frame contains the required `total_energy` and `foces` keys in its metadata. This helps confirm that no corrupted or incomplete structures were accidentally written to the `.extxyz` files, which is essential for successful NequIP training.


In [ ]:
from ase.io import read

frames_train = read("data/al_train.extxyz", ":")
frames_val   = read("data/al_val.extxyz", ":")

missing_train = [i for i,f in enumerate(frames_train) if "total_energy" not in f.info]
missing_val   = [i for i,f in enumerate(frames_val) if "total_energy" not in f.info]

print("Train frames missing total_energy:", missing_train)
print("Val frames missing total_energy:", missing_val)


In [ ]:
from ase.io import read

frames_train = read("data/al_train.extxyz", ":")
frames_val   = read("data/al_val.extxyz", ":")

missing_forces_train = [i for i, f in enumerate(frames_train) if "forces" not in f.arrays]
missing_forces_val   = [i for i, f in enumerate(frames_val) if "forces" not in f.arrays]

print("Train frames missing forces:", missing_forces_train)
print("Val frames missing forces:", missing_forces_val)


Then we load the first structure from the `al_train.extxyz` file and
print two key quantities stored in the dataset:

- **`total_energy`** — the reference potential energy computed with EMT  
- **`forces`** — the atomic forces acting on each atom in the configuration  


In [ ]:
from ase.io import read
frames = read("data/al_train.extxyz", ":")


In [ ]:
print("Energy:", frames[0].info["total_energy"])
print("Forces:\n", frames[0].arrays["forces"])


The `forces` array contains **one 3D force vector per atom** in the structure. Each row represents the force acting on a specific atom along the **x, y, and z** directions (in eV/Å).


## 2.4 NequIP Training Configuration


## NequIP Training Configuration

Below is a summary of the parameters used in the `config.yaml` file, grouped by category:

### **General Settings**
| Parameter | Value | Description |
|----------|--------|-------------|
| `root` | `"."` | Directory where training outputs are stored |
| `run_name` | `"al_nequip_run"` | Folder name for this training run |
| `seed` | `0` | Random seed for reproducibility |
| `default_dtype` | `"float64"` | Numerical precision used in the model |
| `device` | `"cpu"` | Computation device (CPU/GPU) |

### **Dataset Settings**
| Parameter | Value | Description |
|----------|--------|-------------|
| `dataset` | `"ase"` | Tells NequIP to use an ASE dataset |
| `dataset_file_name` | `"data/al_train.extxyz"` | Training data file |
| `validation_dataset_file_name` | `"data/al_val.extxyz"` | Validation data file |
| `ase_args` | `{format: extxyz}` | File format specification |
| `chemical_symbol_to_type` | `{Al: 0}` | Assigns element Al → type 0 |
| `include_keys` | `["total_energy", "forces"]` | Data fields to import from each frame |
| `key_mapping` | `{"total_energy": "total_energy", "forces": "forces"}` | Maps ASE names → internal names |
| `energy_key` | `"total_energy"` | Field used as the energy target |
| `forces_key` | `"forces"` | Field used as the forces target |
| `r_max` | `5.0` | Cutoff radius (Å) for neighbor interactions |
| `batch_size` | `4` | Number of samples per batch |
| `shuffle` | `True` | Whether to shuffle data before training |
| `n_train` | `"80%"` | Fraction of data used for training |
| `n_val` | `"20%"` | Fraction of data used for validation |

### **Model Hyperparameters**
| Parameter | Value | Description |
|----------|--------|-------------|
| `num_features` | `32` | Width of feature channels |
| `l_max` | `1` | Maximum angular momentum used |
| `parity` | `"o3_full"` | Parity handling in equivariant tensors |
| `nonlinearity_type` | `"gate"` | Type of nonlinear activation |
| `n_interactions` | `2` | Number of interaction blocks |

### **Loss Function**
| Parameter | Value | Description |
|----------|--------|-------------|
| `loss_coeffs.total_energy` | `1.0` | Weight for energy loss |
| `loss_coeffs.forces` | `10.0` | Weight for forces loss |
| `per_atom_energy` | `False` | Energy treated as total, not per atom |

### **Training Settings**
| Parameter | Value | Description |
|----------|--------|-------------|
| `max_epochs` | `20` | Maximum number of training epochs |
| `patience` | `5` | Early stopping patience |
| `lr_initial` | `3e-3` | Initial learning rate |
| `optimizer` | `"Adam"` | Optimization algorithm |

---



In [ ]:
config = {
    # ---- general ----
    "root": ".",
    "run_name": "al_nequip_run",
    "seed": 0,
    "default_dtype": "float64",
    "device": "cpu",

    # ---- dataset ----
    "dataset": "ase",
    "dataset_file_name": "data/al_train.extxyz",
    "validation_dataset_file_name": "data/al_val.extxyz",
    "ase_args": {"format": "extxyz"},
    "chemical_symbol_to_type": {"Al": 0},

    # tell NequIP to actually import these from Atoms
    "include_keys": ["total_energy", "forces"],

    # (mapping is trivial here, but harmless)
    "key_mapping": {
        "total_energy": "total_energy",
        "forces": "forces",
    },

    "energy_key": "total_energy",
    "forces_key": "forces",

    "r_max": 5.0,
    "batch_size": 4,
    "shuffle": True,

    # use ALL frames from al_train.extxyz
    "n_train": "80%",
    "n_val": "20%",

    # ---- model hyperparameters ----
    "num_features": 32,
    "l_max": 1,
    "parity": "o3_full",
    "nonlinearity_type": "gate",
    "n_interactions": 2,

    # ---- loss ----
    "loss_coeffs": {
        "total_energy": 1.0,
        "forces": 10.0,
    },
    "per_atom_energy": False,

    # ---- training ----
    "max_epochs": 20,
    "patience": 5,
    "lr_initial": 3e-3,
    "optimizer": "Adam",
}
import yaml
with open("config.yaml", "w") as f:
    yaml.dump(config, f)

print(open("config.yaml").read())


**Configuration Summary**

This YAML file defines all parameters used for training the NequIP model.  
It includes:

- **Dataset settings** (training/validation files, ASE format, included keys)  
- **Model architecture** (number of features, interactions, symmetry settings)  
- **Training hyperparameters** (batch size, learning rate, optimizer, epochs)  
- **Energy/force keys** to correctly map values from the `.extxyz` dataset  

These settings fully specify how NequIP loads the data, builds the model, and performs training.


## 2.6 Training NequIP model on Al data

Confirm that our processed dataset has been successfully created.  
We see the two expected files:

- **al_train.extxyz** — training set
- **al_val.extxyz** — validation set

These files will be used by NequIP during model training.


In [ ]:
!ls data


Then list all files created by NequIP during training.  
You will notice that **another `config.yaml` appears inside this folder**.

This is expected:  
- NequIP **copies your original configuration file** into the run directory (`al_nequip_run/`) to **store the exact settings used during training**.  
- This guarantees **reproducibility**, since even if you later modify your original `config.yaml`, the model folder still contains the precise configuration used to train that run.

In summary:  
- `config.yaml` in the main directory → the file you created and used to start training  
- `config.yaml` inside `al_nequip_run/` → NequIP’s internal copy used for the completed training session


In [ ]:
!ls


shows the following items:

- **`al_nequip_run/`** – the main output directory created by NequIP.  
  It stores everything from the training session, including:
  - the copied `config.yaml` used for training,
  - `best_model.pth` and `last_model.pth`,
  - training logs and metrics.

- **`processed_dataset_*`** – automatically generated cache folders.  
  NequIP preprocesses the ASE dataset once and stores the processed tensors here to speed up future runs.

- **`config.yaml`** – the configuration file *you* created that defines how training should run.

- **`data/`** – contains the dataset files you generated:  
  - `al_train.extxyz`  
  - `al_val.extxyz`



Launches the NequIP training process using the settings defined in `config.yaml`.  
The `--warn-unused` flag tells NequIP to notify us if any configuration keys are not recognized or used, which helps catch typos or unnecessary parameters.


In [ ]:
!nequip-train --warn-unused config.yaml


## NequIP Training Summary

The model was successfully trained for **20 epochs** using our Al dataset generated with EMT.  
During training, NequIP reports detailed metrics for both the **training** and **validation** sets, allowing us to monitor learning quality, force prediction accuracy, and energy error.  
The printed logs show the evolution of the model’s loss, force MAE/RMSE, and energy MAE/RMSE as the network improves over time.

---

## Meaning of Training Metrics

| Metric | Meaning |
|--------|---------|
| **loss** | Total training/validation loss for the batch or epoch. It combines force and energy losses using the specified loss coefficients. |
| **loss_f** | Loss contribution from **forces** only. Since forces usually dominate training, this term is crucial. |
| **loss_e** | Loss contribution from **energies** only. Typically smaller because energies are single values per frame. |
| **f_mae** | Mean Absolute Error (MAE) of predicted forces (eV/Å). Lower is better. |
| **f_rmse** | Root Mean Square Error of forces. Highlights large deviations. |
| **e_mae** | MAE of predicted total energies (eV). |
| **e_rmse** | RMSE of predicted energies. |
| **wal** | Wall-clock time (seconds) spent so far. |
| **LR** | Learning rate used in the current epoch. |

---

## Final Training Results

By the end of epoch **20**, validation errors improved significantly:

- **Force MAE** reached *~0.020–0.030 eV/Å*  
- **Energy MAE** reached *~0.006–0.010 eV*  
- The best validation loss was recorded at epoch **20** with a final value of **0.033**  
- The model stored in `best_model.pth` corresponds to this best validation performance

These values are **reasonable and consistent** for a lightweight demonstration model trained on a small dataset.
Force errors especially are impressively low, showing that NequIP learned local atomic environments well, even with limited structural diversity.

##2.7 Build deployable NequIP model

In [ ]:
!ls al_nequip_run/

These files contain all information needed to analyze, reproduce, and deploy the trained NequIP model.

- **`config.yaml`**  
  A *copy* of the exact configuration NequIP used during training.  
  This file is frozen after training begins, ensuring reproducibility.

- **`best_model.pth`**  
  The model checkpoint with the lowest validation loss.  
  This is the file we use for deployment and as an ASE calculator.

- **`last_model.pth`**  
  The final model from the last epoch, regardless of quality.  
  Useful for debugging but not ideal for deployment.

- **`trainer.pth`**  
  Internal trainer state (optimizer, scheduler, epoch counters).  
  Enables restarting a training run.

- **`log/`**  
  Directory containing logs generated during training.

- **`metrics_batch_train.csv`**, **`metrics_batch_val.csv`**,  
  **`metrics_epoch.csv`**, **`metrics_initialization.csv`**  
  CSV files storing loss curves, validation metrics, force errors, etc.  
  These are useful for plotting and performance analysis.



Then we **deploy** the NequIP model, by converting the trained neural network (from many internal training files) into a **single compact file** (`deployed_model.pth`) that can be used easily in simulations.


In [ ]:
!nequip-deploy build --train-dir al_nequip_run deployed_model.pth


## 2.8 Use trained NequIP model as ASE calculator


**Using the Deployed NequIP Model for MD and Structure Relaxations**

In this step, we load the **deployed NequIP model** and use it as an ASE calculator to run molecular dynamics and geometry optimizations. This demonstrates how the trained network replaces a classical potential (like EMT) and can now predict energies and forces directly.

We evaluate the model on three test systems:

1. **Bulk fcc Aluminum (MD test)**  
   - A short Langevin MD simulation at 300 K is run using NequIP forces.  
   - We print the total energy every few steps to verify the model behaves stably.

2. **Relaxation of an Al(111) surface slab**  
   - A 3×3×4 Al(111) slab is created and relaxed using BFGS.  
   - The final slab energy is computed with the NequIP calculator.

3. **Relaxation of an Al adatom on Al(111)**  
   - An additional Al atom is placed on the slab (ontop site).  
   - The system is relaxed, and the final adatom–slab energy is evaluated.




In [ ]:
from nequip.ase import NequIPCalculator
from ase.build import bulk, fcc111, add_adsorbate
from ase.md.langevin import Langevin
from ase.optimize import BFGS
from ase import units

model_path = "deployed_model.pth"   # <-- IMPORTANT
calc_nequip = NequIPCalculator.from_deployed_model(model_path)

# ---------- 8.1 Bulk Al MD with NequIP ----------
bulk_test = bulk("Al", "fcc", a=4.05).repeat((2, 2, 2))
bulk_test.calc = calc_nequip

dyn_test = Langevin(
    bulk_test,
    timestep=1.0 * units.fs,
    temperature_K=300,
    friction=0.02
)

print("Short MD with NequIP on bulk Al:")
for step in range(20):
    dyn_test.run(1)
    if step % 5 == 0:
        e = bulk_test.get_potential_energy()
        print(f"Step {step:3d}  E_tot = {e: .4f} eV")

# ---------- 8.2 Relax an Al(111) slab with NequIP ----------
slab_test = fcc111("Al", size=(3, 3, 4), a=4.05, vacuum=8.0)
slab_test.calc = calc_nequip

opt_slab_nq = BFGS(slab_test, logfile=None)
opt_slab_nq.run(fmax=0.05)

E_slab_nq = slab_test.get_potential_energy()
print(f"NequIP Al(111) slab energy: {E_slab_nq:.3f} eV")

# ---------- 8.3 Al adatom on Al(111) ----------
slab_adatom_test = fcc111("Al", size=(3, 3, 4), a=4.05, vacuum=8.0)
add_adsorbate(slab_adatom_test, "Al", height=2.0, position="ontop")
slab_adatom_test.center(axis=2)
slab_adatom_test.calc = calc_nequip

opt_ad_nq = BFGS(slab_adatom_test, logfile=None)
opt_ad_nq.run(fmax=0.05)

E_slab_ad_nq = slab_adatom_test.get_potential_energy()
print(f"NequIP Al(111)+Al adatom slab energy: {E_slab_ad_nq:.3f} eV")


In this step, we evaluate how well the trained NequIP model reproduces EMT energies **along a real MD trajectory**. We first run a short 100-step molecular dynamics simulation using EMT as the reference potential and record EMT energies at each step. For the *same exact geometries*, we then compute energies using the deployed NequIP model. By comparing the two energy curves—and plotting the energy difference ΔE over time, we assess whether NequIP remains consistent and accurate when exposed to dynamical configurations that were **not explicitly included** in the training trajectory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.md.langevin import Langevin
from ase import units
from nequip.ase import NequIPCalculator

# --- load deployed NequIP model ---
calc_nequip = NequIPCalculator.from_deployed_model("deployed_model.pth")

# --- run a short MD TRAJECTORY with EMT (reference) ---
bulk_ref = bulk("Al", "fcc", a=4.05).repeat((2, 2, 2))
bulk_ref.calc = EMT()

dyn_emt = Langevin(
    bulk_ref,
    timestep=1.0 * units.fs,
    temperature_K=300,
    friction=0.02,
)

nsteps = 100
E_emt_list = []
E_nq_list  = []

for step in range(nsteps):
    dyn_emt.run(1)

    # EMT energy from the MD calculator
    E_emt = bulk_ref.get_potential_energy()
    E_emt_list.append(E_emt)

    # NequIP energy on the SAME geometry (copy to avoid changing calc)
    snap = bulk_ref.copy()
    snap.calc = calc_nequip
    E_nq = snap.get_potential_energy()
    E_nq_list.append(E_nq)

E_emt_arr = np.array(E_emt_list)
E_nq_arr  = np.array(E_nq_list)
E_err     = E_nq_arr - E_emt_arr
steps     = np.arange(nsteps)

# ---------- FIGURE: MD energy comparison ----------
fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Top: energies vs step
axs[0].plot(steps, E_emt_arr, label="EMT")
axs[0].plot(steps, E_nq_arr, label="NequIP", linestyle="--")
axs[0].set_ylabel("Energy (eV)")
axs[0].set_title("EMT vs NequIP energies along EMT MD trajectory")
axs[0].legend()

# Bottom: NequIP - EMT error vs step
axs[1].plot(steps, E_err)
axs[1].set_xlabel("MD step")
axs[1].set_ylabel("ΔE (NequIP - EMT) [eV]")

plt.tight_layout()
plt.show()


* The comparison between EMT and NequIP along the MD trajectory shows that the learned potential closely reproduces the reference energies at every step. In the top panel, the EMT and NequIP energy curves almost overlap, demonstrating high fidelity and smooth stability of the model under dynamical evolution.

* The bottom panel shows that the energy deviation ΔE remains extremely small and decreases slightly over time, indicating that NequIP generalizes well to the sampled configurations and introduces no noticeable drift.


**Bulk MD with NequIP**

- The total energy during the short MD run stays close to zero:
  - from **-0.0047 eV** to **-0.0033 eV** over 20 steps.  
- This indicates **stable dynamics**, as the model gives consistent energies and does not blow up, which is what we want for a learned potential.

**Surface and Adatom Energies**

- **Al(111) slab energy:** `3.371 eV`  
- **Al(111) + Al adatom energy:** `3.213 eV`  

The adatom system is slightly **lower in total energy** than the clean slab in this setup, which is consistent with the idea that adsorption can be energetically favorable.  
However, remember that the model was trained on **EMT data** and on a **very small, toy dataset**, so these absolute values are **not physically accurate**, but they do show that the NequIP network behaves sensibly and can be used as a drop-in ASE calculator.

**Comparing NequIP Predictions Against EMT**

In this step, we directly compare the predictions of our **trained NequIP model** with the **EMT reference potential** that was used to generate the training data.

Here is what the code does:

1. **Load a validation structure** from `al_val.extxyz`.  
   This ensures we evaluate the model on data it has *not* seen during training.

2. **Compute NequIP predictions**  
   - `E_nq` → total energy predicted by NequIP  
   - `F_nq` → atomic forces predicted by NequIP  

3. **Compute EMT predictions**  
   Using the same atomic geometry, we evaluate:  
   - `E_emt` → EMT total energy  
   - `F_emt` → EMT forces  

4. **Energy error:**  
   `ΔE = E_nq - E_emt`  
   A small value indicates that the neural network has learned the reference potential accurately.

5. **Force error:**  
   `max |ΔF|` computes the *largest* difference in force magnitude between NequIP and EMT across all atoms.  
   Forces are more sensitive than energies, so this is an important indicator of model quality.



In [ ]:
from ase.calculators.emt import EMT
from ase.io import read
import numpy as np

# take one of your training/validation frames
atoms_nq = read("data/al_val.extxyz", 0)
atoms_nq.calc = calc_nequip
E_nq = atoms_nq.get_potential_energy()
F_nq = atoms_nq.get_forces()

atoms_emt = atoms_nq.copy()
atoms_emt.calc = EMT()
E_emt = atoms_emt.get_potential_energy()
F_emt = atoms_emt.get_forces()

print(f"ΔE (NequIP - EMT): {E_nq - E_emt:.2f} eV")

F_diff = F_nq - F_emt
max_fdiff = np.linalg.norm(F_diff, axis=1).max()
print(f"max |ΔF| (eV/Å): {max_fdiff:.2f}")



NequIP reproduces the EMT reference very closely, with only a ~0.01 eV difference in total energy and a maximum force deviation of ~0.07 eV/Å.
These small errors show that the model successfully learned the EMT potential for the structures included in the training set.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from ase.calculators.emt import EMT
from nequip.ase import NequIPCalculator

# --- load deployed NequIP model ---
calc_nequip = NequIPCalculator.from_deployed_model("deployed_model.pth")

# --- load validation structures ---
frames = read("data/al_val.extxyz", ":")

E_nq_list, E_emt_list = [], []
F_nq_list, F_emt_list = [], []

for at in frames:
    # --- NequIP ---
    at_nq = at.copy()
    at_nq.calc = calc_nequip
    E_nq_list.append(at_nq.get_potential_energy())
    F_nq_list.append(at_nq.get_forces())

    # --- EMT ---
    at_emt = at.copy()
    at_emt.calc = EMT()
    E_emt_list.append(at_emt.get_potential_energy())
    F_emt_list.append(at_emt.get_forces())

# Convert to arrays
E_nq = np.array(E_nq_list)
E_emt = np.array(E_emt_list)

# Flatten all force components
F_nq_flat = np.vstack(F_nq_list).reshape(-1)
F_emt_flat = np.vstack(F_emt_list).reshape(-1)

# -----------------------------
#        Parity plots
# -----------------------------
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# --- Energy parity plot ---
ax = axs[0]
ax.scatter(E_emt, E_nq, s=8, alpha=0.6)
emin = min(E_emt.min(), E_nq.min())
emax = max(E_emt.max(), E_nq.max())
ax.plot([emin, emax], [emin, emax])
ax.set_xlabel("EMT energy (eV)")
ax.set_ylabel("NequIP energy (eV)")
ax.set_title("Energy parity plot")

# --- Force parity plot ---
ax = axs[1]
ax.scatter(F_emt_flat, F_nq_flat, s=2, alpha=0.4)
fmin = min(F_emt_flat.min(), F_nq_flat.min())
fmax = max(F_emt_flat.max(), F_nq_flat.max())
ax.plot([fmin, fmax], [fmin, fmax])
ax.set_xlabel("EMT force components (eV/Å)")
ax.set_ylabel("NequIP force components (eV/Å)")
ax.set_title("Force parity plot")

plt.tight_layout()
plt.show()


Both parity plots show that NequIP closely reproduces EMT energies and forces

In this step, we estimate a simple *adsorption energy* for an Al adatom on an Al(111) surface using the NequIP-trained potential.  
First, a bulk fcc Al supercell (32 atoms) is evaluated with NequIP to obtain a **bulk per-atom reference energy**, which acts as an approximate chemical potential for Aluminum.  
Then, using the previously computed NequIP slab energy and slab+adatom energy, we calculate:

$\
E_{\text{ads}} = E_{\text{slab+adatom}} - E_{\text{slab}} - E_{\text{atom}}
\
$

where $E_{\text{atom}}$ is approximated using the bulk per-atom energy.  



In [ ]:
from ase.build import bulk
import numpy as np

# Assumes you already have:
#   calc_nequip
#   E_slab_nq
#   E_slab_ad_nq

# 1) Compute NequIP bulk per-atom energy
bulk_ref = bulk("Al", "fcc", a=4.05).repeat((2, 2, 2))  # 32 atoms
bulk_ref.calc = calc_nequip

E_bulk_nq = bulk_ref.get_potential_energy()
E_atom_nq = E_bulk_nq / len(bulk_ref)   # "chemical potential" of Al in bulk

print(f"NequIP bulk total energy:     {E_bulk_nq:.2f} eV")
print(f"NequIP bulk per-atom energy: {E_atom_nq:.2f} eV")

# 2) "Adsorption energy" using bulk per-atom reference
E_ads_nq = E_slab_ad_nq - E_slab_nq - E_atom_nq
print(f"NequIP Al adsorption energy (ontop): {E_ads_nq:.2f} eV")


The NequIP bulk energy is very close to zero, which is expected because the model was trained on EMT-generated data where absolute energies are not physically meaningful. The adsorption energy of about –0.11 eV indicates a weakly stabilizing interaction, consistent with the limited and demonstration-level dataset used to train the model.